In [2]:
import pandas as pd
import numpy as np
import mlflow
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from sklearn.metrics import( roc_auc_score, average_precision_score,roc_curve,classification_report,confusion_matrix)
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("fraud_detection_xgboost")


2026/05/10 08:22:04 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection_xgboost' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Users/bijal/project1/fraud_detection_api/notebooks/mlruns/610630243242338833', creation_time=1778381524765, experiment_id='610630243242338833', last_update_time=1778381524765, lifecycle_stage='active', name='fraud_detection_xgboost', tags={}, trace_location=None, workspace='default'>

In [3]:
print("="*60)
print("XGBOOST TRAINING WITH MLFLOW")
print("="*60)

XGBOOST TRAINING WITH MLFLOW


In [4]:
def load_data():
    """Load preprocessed data"""
    print("\n📂 Loading preprocessed data...")
    
    X_train = pd.read_csv('data/X_train.csv')
    X_val = pd.read_csv('data/X_val.csv')
    X_test = pd.read_csv('data/X_test.csv')
    y_train = pd.read_csv('data/y_train.csv').squeeze()
    y_val = pd.read_csv('data/y_val.csv').squeeze()
    y_test = pd.read_csv('data/y_test.csv').squeeze()
    
    print(f"✅ Train: {X_train.shape} (fraud: {y_train.mean():.4%})")
    print(f"✅ Validation: {X_val.shape} (fraud: {y_val.mean():.4%})")
    print(f"✅ Test: {X_test.shape} (fraud: {y_test.mean():.4%})")
    
    return X_train, X_val, X_test, y_train, y_val, y_test




In [5]:
def calculate_metrics(model, X, y, split_name):
    """Calculate all relevant metrics"""
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    metrics = {
        'roc_auc': roc_auc_score(y, y_pred_proba),
        'pr_auc': average_precision_score(y, y_pred_proba),
        'accuracy': (y_pred == y).mean(),
        'fraud_recall': (y_pred[y==1] == 1).mean() if (y==1).sum() > 0 else 0,
        'fraud_precision': (y_pred[y_pred==1] == 1).mean() if (y_pred==1).sum() > 0 else 0
    }
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    metrics['true_negatives'] = int(tn)
    metrics['false_positives'] = int(fp)
    metrics['false_negatives'] = int(fn)
    metrics['true_positives'] = int(tp)
    metrics['false_positive_rate'] = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n📊 {split_name} Metrics:")
    print(f"   ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"   PR-AUC: {metrics['pr_auc']:.4f}")
    print(f"   Fraud Recall: {metrics['fraud_recall']:.2%}")
    print(f"   Fraud Precision: {metrics['fraud_precision']:.2%}")
    print(f"   False Positive Rate: {metrics['false_positive_rate']:.4%}")
    
    return metrics


In [6]:
def plot_feature_importance(model, feature_names, save_path):
    """Plot XGBoost feature importance"""
    # Get feature importance (weight = number of times feature used in splits)
    importance = model.feature_importances_
    
    # Create dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False).head(20)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0, 1, len(importance_df)))
    bars = ax.barh(range(len(importance_df)), importance_df['importance'].values, color=colors)
    
    ax.set_yticks(range(len(importance_df)))
    ax.set_yticklabels(importance_df['feature'].values)
    ax.set_xlabel('Feature Importance (Freq)', fontsize=12, fontweight='bold')
    ax.set_title('XGBoost - Top 20 Feature Importances', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, importance_df['importance'].values)):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
                va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Feature importance plot saved to: {save_path}")
    return importance_df

In [7]:
def plot_confusion_matrix(y_true, y_pred, save_path):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legitimate', 'Fraudulent'],
                yticklabels=['Legitimate', 'Fraudulent'])
    
    ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
    ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
    ax.set_title('Confusion Matrix - XGBoost', fontsize=14, fontweight='bold')
    
    # Add annotations
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.1, f'TN: {tn:,} | FP: {fp:,} | FN: {fn:,} | TP: {tp:,}',
            transform=ax.transAxes, ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Confusion matrix saved to: {save_path}")


In [8]:
def train_xgboost():
    """Main training function"""
    
    # Load data
    X_train, X_val, X_test, y_train, y_val, y_test = load_data()
    
    # Calculate class imbalance ratio for scale_pos_weight
    ratio = len(y_train[y_train==0]) / len(y_train[y_train==1])
    print(f"\n⚖️ Class imbalance ratio: {ratio:.1f}:1")
    
    # XGBoost parameters
    params = {
        'n_estimators': 200,
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': ratio,  # Handles class imbalance
        'random_state': 42,
        'eval_metric': 'aucpr',  # Use PR-AUC for evaluation
        'use_label_encoder': False,
        'verbosity': 0,
        'min_child_weight': 1,
        'gamma': 0
    }
    
    print("\n🚀 Starting MLflow run...")
    
    # Start MLflow run
    with mlflow.start_run(run_name="XGBoost_Fraud_Detection") as run:
        
        print("\n📝 Logging parameters...")
        mlflow.log_params(params)
        
        print("\n🏋️ Training XGBoost model...")
        model = xgb.XGBClassifier(**params)
        
        # Train with evaluation set for monitoring
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        print("✅ Model training complete!")
        
        # Calculate metrics for all splits
        print("\n📊 Calculating metrics...")
        train_metrics = calculate_metrics(model, X_train, y_train, "Training")
        val_metrics = calculate_metrics(model, X_val, y_val, "Validation")
        test_metrics = calculate_metrics(model, X_test, y_test, "Test")
        
        # Log metrics to MLflow
        print("\n📝 Logging metrics to MLflow...")
        for metric_name, value in val_metrics.items():
            if isinstance(value, (int, float)):
                mlflow.log_metric(f"val_{metric_name}", value)
        
        for metric_name, value in test_metrics.items():
            if isinstance(value, (int, float)):
                mlflow.log_metric(f"test_{metric_name}", value)
        
        # Plot and log feature importance
        print("\n🎨 Generating feature importance plot...")
        importance_df = plot_feature_importance(
            model, X_train.columns, 
            'data/xgboost_feature_importance.png'
        )
        
        # Save feature importance as CSV
        importance_df.to_csv('data/xgboost_feature_importance.csv', index=False)
        
        # Log artifacts to MLflow
        mlflow.log_artifact('data/xgboost_feature_importance.png')
        mlflow.log_artifact('data/xgboost_feature_importance.csv')
        
        # Plot and log ROC/PR curves
        print("\n🎨 Generating performance curves...")
        plot_roc_pr_curves(model, X_val, y_val, 'data/xgboost_performance_curves.png')
        mlflow.log_artifact('data/xgboost_performance_curves.png')
        
        # Plot and log confusion matrix
        print("\n🎨 Generating confusion matrix...")
        y_val_pred = (model.predict_proba(X_val)[:, 1] >= 0.5).astype(int)
        plot_confusion_matrix(y_val, y_val_pred, 'data/xgboost_confusion_matrix.png')
        mlflow.log_artifact('data/xgboost_confusion_matrix.png')
        
        # Save the model
        print("\n💾 Saving model...")
        os.makedirs('models', exist_ok=True)
        
        # Save with joblib
        joblib.dump(model, 'models/xgboost_model.pkl')
        print(f"✅ Model saved to: models/xgboost_model.pkl")
        
        # Log model to MLflow
        mlflow.xgboost.log_model(model, "xgboost_model")
        
        # Save all metrics to JSON
        all_metrics = {
            'train': {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in train_metrics.items()},
            'val': {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in val_metrics.items()},
            'test': {k: float(v) if isinstance(v, (np.floating, float)) else v for k, v in test_metrics.items()},
            'params': params,
            'run_id': run.info.run_id
        }
        
        with open('models/xgboost_metrics.json', 'w') as f:
            json.dump(all_metrics, f, indent=2)
        print(f"✅ Metrics saved to: models/xgboost_metrics.json")
        
        # Print summary
        print("\n" + "="*60)
        print("TRAINING COMPLETE! SUMMARY")
        print("="*60)
        print(f"\n🏆 XGBoost Performance:")
        print(f"   Validation PR-AUC: {val_metrics['pr_auc']:.4f}")
        print(f"   Validation ROC-AUC: {val_metrics['roc_auc']:.4f}")
        print(f"   Validation Fraud Recall: {val_metrics['fraud_recall']:.2%}")
        print(f"   Validation False Positive Rate: {val_metrics['false_positive_rate']:.4%}")
        
        print(f"\n📊 Test Performance (Final):")
        print(f"   Test PR-AUC: {test_metrics['pr_auc']:.4f}")
        print(f"   Test Fraud Recall: {test_metrics['fraud_recall']:.2%}")
        
        print(f"\n🔗 MLflow Run ID: {run.info.run_id}")
        print(f"📁 View all runs with: mlflow ui")
        print(f"🌐 Then open: http://localhost:5000")
        
        # Print top features
        print("\n📈 Top 10 Most Important Features:")
        for i, row in importance_df.head(10).iterrows():
            print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")
        
        return model, all_metrics

In [9]:
def load_and_predict_example():
    """Example: Load saved model and make prediction"""
    print("\n" + "="*60)
    print("LOADING MODEL FOR INFERENCE EXAMPLE")
    print("="*60)
    
    # Load model
    model = joblib.load('models/xgboost_model.pkl')
    print("✅ Model loaded successfully")
    
    # Load a sample transaction from test set
    X_test = pd.read_csv('data/X_test.csv')
    
    # Take first transaction
    sample = X_test.iloc[0:1]
    
    # Make prediction
    proba = model.predict_proba(sample)[0][1]
    prediction = "FRAUD" if proba > 0.5 else "LEGIT"
    
    print(f"\n🔮 Sample Prediction:")
    print(f"   Fraud Probability: {proba:.2%}")
    print(f"   Prediction: {prediction}")
    
    return model

In [11]:
if __name__ == "__main__":
    # Train the model
    model, metrics = train_xgboost()
    
    # Optional: Test inference
    load_and_predict_example()


📂 Loading preprocessed data...


FileNotFoundError: [Errno 2] No such file or directory: 'data/X_train.csv'